# Assignment 3 — AnnotationAgent

Этот ноутбук показывает weak supervision + HITL для `ru_fp_bench`:

- автоматическую разметку;
- confidence-aware review queue;
- генерацию `annotation_spec.md`;
- экспорт в Label Studio.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "agents").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from agents.annotation_agent import AnnotationAgent

candidate_paths = [
    ROOT / "data/interim/clean.parquet",
    ROOT / "data/raw/merged_raw.csv",
]
for path in candidate_paths:
    if path.exists():
        df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
        break
else:
    raise FileNotFoundError("No input dataset found in data/interim/clean.parquet or data/raw/merged_raw.csv")

agent = AnnotationAgent(modality="text", config=ROOT / "config.yaml")
df_labeled = agent.auto_label(df)
df_labeled[[c for c in ["text", "predicted_label", "confidence", "label_reason"] if c in df_labeled.columns]].head()

In [ ]:
import matplotlib.pyplot as plt

metrics = agent.check_quality(df_labeled)
metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_labeled["predicted_label"].astype(str).value_counts().plot(kind="bar", ax=axes[0], color="coral")
axes[0].set_title("Predicted label distribution")
axes[0].set_ylabel("count")

df_labeled["confidence"].astype(float).plot(kind="hist", bins=20, ax=axes[1], color="steelblue")
axes[1].axvline(agent.confidence_threshold, color="red", linestyle="--", label="threshold")
axes[1].set_title("Confidence distribution")
axes[1].set_xlabel("confidence")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
spec_path = agent.generate_spec(df_labeled)
labelstudio_path = agent.export_to_labelstudio(df_labeled)
review_queue = agent.build_review_queue(df_labeled)

print(spec_path)
print(labelstudio_path)
review_queue.head() if review_queue is not None else "No low-confidence rows"

## Что показать на защите

- распределение `predicted_label`;
- среднюю confidence и количество low-confidence примеров;
- примеры `label_reason`;
- как HITL queue помогает отправить сомнительные тексты на ручную проверку.